# `setup_inicial` — carga histórica

Monta a base do zero (ou completa uma existente) para o intervalo **`INICIO` .. `FIM`** que
**você define na primeira célula**. O boletim e todo o cálculo rodam para **todos os dias úteis**
do intervalo — não só as pontas.

**Fontes com histórico curto** (deb, NTN-B, curva DI, CRI/CRA) não entregam o intervalo inteiro.
Para elas o notebook já pede **o máximo que a fonte guarda** (janelas fixas na célula de config) —
você não precisa mexer.

**Cada bloco:** roda o fluxo → confere no `.db` (contagem + quantos pregões ficaram cobertos).

**Não apaga dados.** Tudo é `CREATE TABLE IF NOT EXISTS` + UPSERT. Rodar de novo por cima de uma
base populada é seguro. Bloco que falhar: corrija e **re-rode só ele** (tudo idempotente).

⚠️ **Demorado** (boletim e cálculo pregão a pregão). Dá `Run All` e volta depois.
⚠️ **Rode a partir da pasta `code/`.**

## Config — a janela da carga
**Só `INICIO` e `FIM` são seus.** O resto são os limites reais de cada fonte.

In [ ]:
import sqlite3
import sys
import time
from datetime import date, timedelta
from pathlib import Path

if not (Path.cwd() / "scripts").exists():
    raise SystemExit(f"Rode a partir da pasta code/. cwd atual: {Path.cwd()}")
sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd() / "scripts"))
import pipeline_core as pc
from lib.db import ObterBanco

# ==================== EDITE AQUI — janela da carga ====================
INICIO = date(2026, 1, 2)     # 1o dia do historico que voce quer
FIM    = date(2026, 7, 7)     # ultimo dia (nao passe de HOJE)
# ======================================================================

# Limites REAIS de cada fonte (HARDCODED — nao adianta pedir alem disso).
# Sao contados a partir de FIM_SRC; a carga puxa o maximo que cada fonte ainda guarda.
DIAS_DEB_NTNB = 125   # deb + NTN-B: XLS Anbima fica ~4 meses no ar (dias corridos)
DIAS_CURVA_DI = 20    # curva DI B3: API guarda ~20 pregoes
DIAS_CRICRA   = 5     # CRI/CRA: portal Anbima guarda ~5 pregoes

WORKERS = 8           # threads das chamadas de API (calc_taxa / ntnb)

ObterBanco().close()      # cria/verifica schema — NAO apaga dados
DB   = "data/trades.db"
HOJE = date.today()

if FIM > HOJE:
    raise SystemExit(f"FIM ({FIM}) e futuro. Use no maximo {HOJE}.")
if INICIO > FIM:
    raise SystemExit(f"INICIO ({INICIO}) depois de FIM ({FIM}).")

DIAS     = [d.isoformat() for d in pc.DiasUteisEntre(INICIO, FIM)]   # TODOS os pregoes da janela
FIM_SRC  = min(FIM, pc.DiaUtilAnterior(HOJE))   # fontes so publicam ate o ultimo pregao fechado
INI_DEB  = max(INICIO, FIM_SRC - timedelta(days=DIAS_DEB_NTNB))
DIAS_DI  = [d.isoformat() for d in pc.UltimosNDiasUteis(DIAS_CURVA_DI, ref=FIM_SRC)]
DIAS_CR  = [d.isoformat() for d in pc.UltimosNDiasUteis(DIAS_CRICRA,   ref=FIM_SRC)]

if not DIAS:
    raise SystemExit("Intervalo sem dias uteis — confira INICIO/FIM.")

print(f"Base: {Path(DB).resolve()}")
print(f"Janela pedida : {INICIO} .. {FIM}  ->  {len(DIAS)} pregoes ({DIAS[0]} ... {DIAS[-1]})")
print("\nO que cada fonte vai realmente entregar:")
print(f"  boletim B3        : {DIAS[0]} .. {DIAS[-1]}   ({len(DIAS)} pregoes)")
print(f"  anbima deb        : {INI_DEB} .. {FIM_SRC}   (~{DIAS_DEB_NTNB}d, limite da fonte)")
print(f"  anbima NTN-B      : {INI_DEB} .. {FIM_SRC}   (~{DIAS_DEB_NTNB}d, limite da fonte)")
print(f"  curva DI B3       : {DIAS_DI[0]} .. {DIAS_DI[-1]}   ({len(DIAS_DI)} pregoes, limite da fonte)")
print(f"  anbima CRI/CRA    : {DIAS_CR[0]} .. {DIAS_CR[-1]}   ({len(DIAS_CR)} pregoes, limite da fonte)")
print(f"  fianalytics       : snapshot atual (sem data)")
print(f"  anbima data       : universo completo ou incremental (ver bloco 8)")

conferencias = []

def Checar(desc, sql, params=()):
    """Contagem simples: gravou alguma coisa?"""
    try:
        n = sqlite3.connect(DB).execute(sql, params).fetchone()[0]
    except Exception as e:
        print(f"[ERRO ] {desc}: {e}")
        conferencias.append((desc, False))
        return False
    ok = bool(n and n > 0)
    print(f"{'[OK]    ' if ok else '[VAZIO] '}{desc}: {n:,} linha(s)")
    conferencias.append((desc, ok))
    return ok

def Cobertura(desc, tabela, colunaData, dias, extraSql=""):
    """Quantos dos `dias` esperados ficaram com dados na tabela? (o teste que importa)"""
    ini, fim = dias[0], dias[-1]
    sql = (f"SELECT COUNT(DISTINCT {colunaData}) FROM {tabela} "
           f"WHERE {colunaData} BETWEEN ? AND ? {extraSql}")
    try:
        n = sqlite3.connect(DB).execute(sql, (ini, fim)).fetchone()[0]
    except Exception as e:
        print(f"[ERRO ] {desc}: {e}")
        conferencias.append((desc, False))
        return False
    ok = n >= len(dias)
    marca = "[OK]    " if ok else ("[PARCIAL]" if n else "[VAZIO] ")
    print(f"{marca} {desc}: {n}/{len(dias)} pregoes com dados ({ini} .. {fim})")
    conferencias.append((desc, ok))
    return ok

## Scraping — 1 bloco por fonte
Anbima Data (o mais pesado) por último, mas ainda antes do cálculo — o `match_referencias` e os
spreads dependem da `InfoAtivos` que ele preenche.

In [ ]:
# 1. Boletim B3 (negocios) -> NegociosBrutos. Itera pregao a pregao sobre INICIO..FIM.
pc.Boletim(DIAS[0], DIAS[-1])
Cobertura("boletim -> NegociosBrutos", "NegociosBrutos", "dtNegocio", DIAS,
          "AND cdSituacao != 'Cancelado'")

In [ ]:
# 2. FI Analytics planilha (caracteristicas) -> InfoAtivos. Snapshot atual, sem data.
pc.FiAnalytics()
Checar("fianalytics -> InfoAtivos (gravado hoje)",
       "SELECT COUNT(*) FROM InfoAtivos WHERE DATE(dtAtualizacao) = DATE('now','localtime')")

In [ ]:
# 3. Anbima debentures (taxa indicativa) -> AnbimaIndicativos. Janela HARDCODED (~4 meses).
#    Datas fora da janela dao 404 no XLS e sao puladas pelo proprio script — normal.
pc.AnbimaDeb(INI_DEB.isoformat(), FIM_SRC.isoformat())
Cobertura("anbima_deb -> AnbimaIndicativos", "AnbimaIndicativos", "dtReferencia",
          [d.isoformat() for d in pc.DiasUteisEntre(INI_DEB, FIM_SRC)])

In [ ]:
# 4. Anbima NTN-B (MtM) -> MtmAnbima. Janela HARDCODED (~4 meses).
#    Duration em paralelo + skip do ja calculado (re-rodar e barato).
pc.Ntnb(INI_DEB.isoformat(), FIM_SRC.isoformat(), workers=WORKERS)
Cobertura("ntnb -> MtmAnbima (NTN-B)", "MtmAnbima", "dtReferencia",
          [d.isoformat() for d in pc.DiasUteisEntre(INI_DEB, FIM_SRC)],
          "AND cdTicker LIKE 'NTN-B%'")

In [ ]:
# 5. Curva DI B3 (MtM) -> MtmAnbima. Janela HARDCODED: ~20 pregoes (limite da API).
for d in DIAS_DI:
    pc.CurvaDi(d)
Cobertura("curva_di -> MtmAnbima (DI1)", "MtmAnbima", "dtReferencia", DIAS_DI,
          "AND cdTicker LIKE 'DI1%'")

In [ ]:
# 6. Anbima CRI/CRA (taxa indicativa, Playwright) -> AnbimaIndicativos.
#    Janela HARDCODED: ~5 pregoes (o portal so publica os mais recentes).
for d in DIAS_CR:
    pc.AnbimaCriCra(d)
Cobertura("anbima_cricra -> AnbimaIndicativos (CRI/CRA)", "AnbimaIndicativos", "dtReferencia", DIAS_CR,
          "AND (cdTicker LIKE 'CRA%' OR cdTicker GLOB '[0-9]*')")

In [ ]:
# 7. Outstanding via Bloomberg -> Outstanding.  *** SO RODA NO BANCO ***
#    No PC pessoal da [FALHA]/[VAZIO] (sem terminal Bloomberg) — esperado.
pc.Outstanding(DIAS[0], DIAS[-1])
Cobertura("outstanding -> Outstanding (so no banco)", "Outstanding", "dtOutstanding", DIAS)

In [ ]:
# 8. Anbima Data (caracteristicas + agenda) -> InfoAtivos + FluxoAtivos. O passo MAIS PESADO.
#      base VAZIA       -> universo COMPLETO (--mode full), ~4200 debentures + CRI/CRA
#      base JA POPULADA -> so o INCREMENTAL da janela (tickers novos)
FORCAR_FULL = False   # True forca o --mode full mesmo com a base populada

nInfo = sqlite3.connect(DB).execute("SELECT COUNT(*) FROM InfoAtivos").fetchone()[0]
if FORCAR_FULL or nInfo == 0:
    print(f"InfoAtivos = {nInfo:,} -> universo COMPLETO (--mode full)")
    pc.AnbimaData(full=True)
else:
    print(f"InfoAtivos = {nInfo:,} ja populado -> INCREMENTAL {DIAS[0]}..{DIAS[-1]}")
    pc.AnbimaData(DIAS[0], DIAS[-1])

Checar("anbima_data -> InfoAtivos",  "SELECT COUNT(*) FROM InfoAtivos")
Checar("anbima_data -> FluxoAtivos", "SELECT COUNT(*) FROM FluxoAtivos")

## Cálculo — 1 bloco por fluxo, pregão a pregão
Percorre **todos os dias úteis** da janela (`DIAS`). Idempotente: re-rodar pula o já feito.
Ordem obrigatória: taxa → filtrar → spread Anbima → match → spread over → relatório.

In [ ]:
# 9. Taxa por trade (cascata FI Analytics -> B3) -> NegociosProcessados. O passo mais LENTO.
for X in DIAS:
    pc.CalcTaxa(X, workers=WORKERS)
Cobertura("calc_taxa -> NegociosProcessados", "NegociosProcessados", "dtLiquidacao", DIAS,
          "AND vrTaxaCalculada IS NOT NULL")

In [ ]:
# 10. Filtrar (VALIDO / FUNDO / BROKER / PF) -> NegociosProcessados.cdStatus
for X in DIAS:
    pc.Filtrar(X)
Cobertura("filtrar -> cdStatus = VALIDO", "NegociosProcessados", "dtLiquidacao", DIAS,
          "AND cdStatus = 'VALIDO'")

In [ ]:
# 11. Spread Anbima das indicativas -> AnbimaIndicativos.vrSpreadAnbima
#     So faz sentido onde ha indicativa: a janela de deb (INI_DEB..FIM_SRC).
DIAS_IND = [d.isoformat() for d in pc.DiasUteisEntre(INI_DEB, FIM_SRC)]
for X in DIAS_IND:
    pc.SpreadAnbima(X)
Cobertura("spread_anbima -> vrSpreadAnbima", "AnbimaIndicativos", "dtReferencia", DIAS_IND,
          "AND vrSpreadAnbima IS NOT NULL")

In [ ]:
# 12. Match de referencia (global, sem data) -> InfoAtivos.cdReferencia. ANTES do spread_over.
pc.MatchRef()
Checar("match_ref -> cdReferencia preenchido",
       "SELECT COUNT(*) FROM InfoAtivos WHERE cdReferencia IS NOT NULL")

In [ ]:
# 13. Spread over dos trades (casado por dtNegocio) -> NegociosProcessados.vrSpreadOver
for X in DIAS:
    pc.SpreadOver(X)
Cobertura("spread_over -> vrSpreadOver", "NegociosProcessados", "dtLiquidacao", DIAS,
          "AND vrSpreadOver IS NOT NULL")

In [ ]:
# 14. Gerar relatorio HTML (le a base inteira)
alvo  = Path("data/relatorios/relatorio_secundario.html")
antes = alvo.stat().st_mtime if alvo.exists() else 0
pc.Relatorio()
ok = alvo.exists() and alvo.stat().st_mtime > antes
conferencias.append(("relatorio -> HTML regravado", ok))
if alvo.exists():
    print(f"{'[OK]    ' if ok else '[VAZIO] '}relatorio: {alvo.resolve()} "
          f"({alvo.stat().st_size / 1e6:.1f} MB, gravado ha {time.time() - alvo.stat().st_mtime:.0f}s)")
else:
    print("[VAZIO] relatorio: arquivo nao foi criado")

## Conferência final — a base ficou de pé?

In [ ]:
oks = sum(1 for _, ok in conferencias if ok)
print(f"{'#' * 62}\n# CARGA: {oks}/{len(conferencias)} conferencias plenas\n{'#' * 62}")
for desc, ok in conferencias:
    print(f"  {'OK   ' if ok else 'FALTA'}  {desc}")

print("\nTamanho das tabelas:")
c = sqlite3.connect(DB)
for t in ["NegociosBrutos", "NegociosProcessados", "InfoAtivos", "AnbimaIndicativos",
          "MtmAnbima", "FluxoAtivos", "Outstanding"]:
    print(f"  {t:22s}: {c.execute('SELECT COUNT(*) FROM ' + t).fetchone()[0]:>10,} linhas")

print("\nPregoes de negocio na base (ultimos 10):")
for dt, n in c.execute("SELECT dtNegocio, COUNT(*) FROM NegociosBrutos "
                       "GROUP BY dtNegocio ORDER BY dtNegocio DESC LIMIT 10"):
    print(f"  {dt}: {n:>8,}")
c.close()

print("\nFALTA/PARCIAL nas fontes de janela curta (DI, CRI/CRA) e esperado se sua janela"
      " for maior que o que a fonte guarda. Nas demais, re-rode o bloco correspondente.")